# placax: MaskPlace training on Colab

Clones `placax`, installs it with GPU support, auto-detects the largest `n_episodes` buffer size that fits (16GB on the free T4 tier vs. 4GB locally, so this should pick something higher than 1), then runs `scripts/run_maskplace.py`.

Before running: **Runtime > Change runtime type > T4 GPU** (or better, if you have Colab Pro).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# Clone the repo. If it's private, either paste a token below
# (https://github.com/settings/tokens, "repo" scope) or use
# Colab's "Files > mount GitHub" flow instead.
GITHUB_TOKEN = ""  # leave empty for a public repo

repo_url = (
    f"https://{GITHUB_TOKEN}@github.com/paulin-dev/placax.git"
    if GITHUB_TOKEN else
    "https://github.com/paulin-dev/placax.git"
)
!git clone $repo_url
%cd placax

In [ ]:
# Colab's runtime already ships a working CUDA-enabled JAX - installing placax's own
# .[cuda] extra on top of it registers a second, conflicting 'cuda' PJRT plugin (you'd
# see a "PJRT_Api already exists for device type cuda" error at import time). Just
# install placax itself plus the resnet extra and reuse Colab's preinstalled JAX.
#
# -U (upgrade) matters here too: Colab's base image ships some packages (flax in
# particular) old enough to be technically "compatible" with flaxmodels' own loose
# >=0.4.0 requirement, but too old for what jax/flaxmodels actually need at import
# time - pip won't touch an already-satisfied requirement without -U, so without it
# you can hit import-time errors that "pip install -U flax" alone would fix.
!pip install -q -U -e .[resnet]
!pip install -U flax

In [ ]:
import jax
print(jax.devices())  # should list a Gpu/Cuda device, not just Cpu

## (optional) checkpoint to Drive

Colab runtimes are ephemeral - if you want the checkpoint/training log to survive a
disconnect, point the run at a Drive-backed output dir instead of the repo's local one.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import pathlib
    drive_out = pathlib.Path("/content/drive/MyDrive/placax/adaptec1_output_maskplace")
    drive_out.mkdir(parents=True, exist_ok=True)
    local_out = pathlib.Path("benchmarks/adaptec1/output_maskplace")
    if local_out.is_symlink() or local_out.exists():
        import shutil
        if local_out.is_symlink():
            local_out.unlink()
        else:
            shutil.rmtree(local_out)
    local_out.symlink_to(drive_out)

## Auto-detect the largest n_episodes that fits

`--n_episodes=auto` isn't a flag of `run_maskplace.py` (see its own docstring for why): the training process reserves a large chunk of GPU memory for itself just by starting up (JAX's default allocator claims ~75% of the device on first use), so it can't accurately probe candidate buffer sizes in disposable subprocesses without competing with them for the same memory. `scripts/subprocess_search.py` fixes that by never importing jax/placax itself - it's a plain subprocess launcher, so it can run *before* anything GPU-related starts.

It also needs no special cooperation from `run_maskplace.py` - it just runs the script's own ordinary CLI once per candidate `n_episodes` value, stopping at the first one that doesn't fit. `--no_checkpoint` keeps each attempt fast and independent (no state read from or left behind by one candidate affecting the next).

`--eval_every=1`/`--n_iterations=4` here deliberately don't match the real training run's `--eval_every=10` below - the eval rollout is a separately-compiled executable with its own memory footprint, so a probe needs to cross at least one eval boundary (plus a couple more iterations, to catch ordinary GPU allocator fragmentation drift) to be representative at all; forcing it every iteration reaches that footprint by iteration 1 instead of iteration 10, so 4 total iterations suffice instead of 13. The one thing this trades away: JAX's allocator can behave slightly differently depending on the exact iteration pattern (eval every iteration vs. only every 10th), so this is a faster but marginally less exact stand-in for the real run's precise allocator history - a second-order effect on an already-approximate probe.

In [ ]:
!python -m scripts.subprocess_search scripts.run_maskplace '--n_episodes=[1,2,4,8,10]' \
    --benchmark_dir=benchmarks/adaptec1 --macro_budget=128 --eval_every=1 --n_iterations=4 --no_checkpoint \
    2>&1 | tee /tmp/n_episodes_search.log

In [ ]:
N_EPISODES = next(
    line.split("=", 1)[1].strip()
    for line in open("/tmp/n_episodes_search.log")
    if line.startswith("RESULT=")
)
print(f"-> n_episodes={N_EPISODES}")

## Train

Runs the actual MaskPlace pipeline with the `n_episodes` detected above. Re-run this cell to
continue training - it resumes from its own checkpoint automatically.

In [ ]:
!python -m scripts.run_maskplace --n_episodes=10 --n_iterations=300 --eval_every=5 --placement_images --patience=10

## Visualize the result

Renders the final macro placement (and training curves / observation channels) from the
checkpoint just trained above, via `scripts/visualize.py` - `--preset=maskplace` matches
this notebook's training run (macro-budget-truncated netlist, wiremask observation).

In [ ]:
!python -m scripts.visualize --preset=maskplace

from IPython.display import Image, display

display(Image(filename="benchmarks/adaptec1/output_maskplace/final_placement.png"))
display(Image(filename="benchmarks/adaptec1/output_maskplace/training_curves.png"))